# 01 — Logistic Regression baseline

**Owners:** Nanda / Khishan  
**Role:** interpretable linear baseline

Interview explanation: *“Logistic Regression establishes how much fraud signal can
be captured by additive linear effects. Its coefficients are interpretable, and it
provides a justified baseline before adopting more complex nonlinear models.”*


## Feature engineering for this approach

- Real numerical quantities: training median imputation, missing indicators, scaling.
- Low/medium-cardinality categories: rare grouping through `min_frequency`, then sparse one-hot encoding.
- High-cardinality categories and numeric codes such as `card1`: training-only frequency encoding.
- Continuous `V*`, `C*`, `D*`, and numeric `id_*` features are retained; high numerical uniqueness is not a reason to drop continuous measurements.
- `class_weight="balanced"` addresses the 3.5% fraud prevalence without synthetic SMOTE records.

Every learned transformation is inside the saved pipeline, preventing training-serving skew.


In [ ]:
from pathlib import Path
_install_root = Path.cwd().resolve()
for _candidate in [_install_root, *_install_root.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)


In [ ]:
required = [
    PROCESSED_DIR / "train.parquet",
    PROCESSED_DIR / "validation.parquet",
    PROCESSED_DIR / "test.parquet",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run 00_shared_data_preparation.ipynb first. Missing: " + ", ".join(missing)
    )

train = pd.read_parquet(required[0])
validation = pd.read_parquet(required[1])
test = pd.read_parquet(required[2])

def stratified_debug_sample(frame, rows):
    if rows is None or rows >= len(frame):
        return frame
    return (
        frame.groupby("isFraud", group_keys=False)
        .apply(lambda group: group.sample(
            n=max(1, round(rows * len(group) / len(frame))),
            random_state=RANDOM_SEED,
        ), include_groups=True)
        .sort_values(["TransactionDT", "TransactionID"])
        .reset_index(drop=True)
    )

FAST_RUN = False  # Set True only to verify the notebook; never report these metrics.
if FAST_RUN:
    train = stratified_debug_sample(train, 60_000)
    validation = stratified_debug_sample(validation, 20_000)
    test = stratified_debug_sample(test, 20_000)

TARGET = "isFraud"
DROP_FROM_MODEL = ["isFraud", "TransactionID"]
X_train, y_train = train.drop(columns=DROP_FROM_MODEL), train[TARGET].astype("int8")
X_validation, y_validation = validation.drop(columns=DROP_FROM_MODEL), validation[TARGET].astype("int8")
X_test, y_test = test.drop(columns=DROP_FROM_MODEL), test[TARGET].astype("int8")

print("Train:", X_train.shape, "fraud rate:", f"{y_train.mean():.4%}")
print("Validation:", X_validation.shape, "fraud rate:", f"{y_validation.mean():.4%}")
print("Test:", X_test.shape, "fraud rate:", f"{y_test.mean():.4%}")


In [ ]:
MODEL_KEY = "logistic_regression"


In [ ]:
from datetime import datetime, timezone
from src.fraud_pipeline.artifacts import build_manifest, package_versions, write_json
from src.fraud_pipeline.evaluation import evaluate_binary_classifier, select_operating_threshold

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = ARTIFACT_ROOT / MODEL_KEY / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("This run will be saved to:", RUN_DIR)


## Build and train the complete serializable pipeline

The sparse `saga` solver avoids materializing a huge dense one-hot matrix. A GPU is
unnecessary for this model.


In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from src.fraud_pipeline.preprocessing import build_logistic_preprocessor, infer_feature_groups

groups = infer_feature_groups(X_train, low_cardinality_max=100)
print({name: len(columns) for name, columns in groups.items()})
preprocessor = build_logistic_preprocessor(groups, rare_min_count=20)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        C=0.1,
        penalty="l2",
        solver="saga",
        class_weight="balanced",
        max_iter=300,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        verbose=1,
    )),
])

started = time.perf_counter()
pipeline.fit(X_train, y_train)
training_seconds = time.perf_counter() - started
print(f"Training time: {training_seconds / 60:.1f} minutes")


## Select the threshold on validation, evaluate holdout once

PR-AUC is primary because accuracy is misleading for rare fraud. The threshold
maximizes recall while satisfying a configurable minimum validation precision.


In [ ]:
validation_probability = pipeline.predict_proba(X_validation)[:, 1]
threshold_record = select_operating_threshold(
    y_validation.to_numpy(), validation_probability, minimum_precision=0.10
)
threshold = float(threshold_record["threshold"])
validation_metrics = evaluate_binary_classifier(y_validation, validation_probability, threshold)

started = time.perf_counter()
test_probability = pipeline.predict_proba(X_test)[:, 1]
prediction_seconds = time.perf_counter() - started
test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)

display(pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])[
    ["pr_auc", "roc_auc", "precision", "recall", "f1", "brier_score"]
])
print("Selected threshold:", threshold_record)


## Explain coefficients and save the deployment bundle

Positive coefficients increase the fraud log-odds; negative coefficients decrease
them. Association does not prove causality.


In [ ]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
coefficients = pipeline.named_steps["classifier"].coef_[0]
importance = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
importance["absolute_coefficient"] = importance["coefficient"].abs()
importance.sort_values("absolute_coefficient", ascending=False).head(100).to_csv(
    RUN_DIR / "top_coefficients.csv", index=False
)

model_path = RUN_DIR / "model.joblib"
joblib.dump(pipeline, model_path, compress=3)
pd.DataFrame({"TransactionID": validation["TransactionID"], "isFraud": y_validation, "probability": validation_probability}).to_parquet(RUN_DIR / "validation_predictions.parquet", index=False)
pd.DataFrame({"TransactionID": test["TransactionID"], "isFraud": y_test, "probability": test_probability}).to_parquet(RUN_DIR / "test_predictions.parquet", index=False)
write_json(RUN_DIR / "threshold.json", threshold_record)
write_json(RUN_DIR / "metrics.json", {"validation": validation_metrics, "test": test_metrics})
write_json(RUN_DIR / "feature_schema.json", {"model": MODEL_KEY, "groups": groups, "raw_input_contract": "data/processed/raw_input_schema.json"})
write_json(RUN_DIR / "training_config.json", {
    "model": MODEL_KEY, "run_id": RUN_ID, "random_seed": RANDOM_SEED,
    "fast_run": FAST_RUN, "training_seconds": training_seconds,
    "test_prediction_seconds": prediction_seconds,
    "test_rows_per_second": len(X_test) / prediction_seconds,
    "parameters": pipeline.named_steps["classifier"].get_params(),
    "versions": package_versions(["numpy", "pandas", "scikit-learn", "joblib"]),
})


## Mandatory reload test

A model is not complete until its disk artifact reproduces the in-memory scores.
This is the exact operation the future FastAPI backend will perform.


In [ ]:
reloaded = joblib.load(model_path)
before = pipeline.predict_proba(X_validation.iloc[:5])[:, 1]
after = reloaded.predict_proba(X_validation.iloc[:5])[:, 1]
np.testing.assert_allclose(before, after, rtol=1e-7, atol=1e-9)
write_json(RUN_DIR / "manifest.json", build_manifest(RUN_DIR))
print("Reload test passed:", after)
print("Artifact directory:", RUN_DIR)


In [ ]:
# Optional promotion step: upload this versioned run to a private Cloudflare R2 bucket.
# Create these as Lightning secrets/environment variables; never paste keys into a cell.
UPLOAD_TO_R2 = False

if UPLOAD_TO_R2:
    import boto3
    required_names = [
        "R2_ENDPOINT_URL", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME"
    ]
    absent = [name for name in required_names if not os.getenv(name)]
    if absent:
        raise RuntimeError("Missing Lightning secrets: " + ", ".join(absent))
    client = boto3.client(
        "s3",
        endpoint_url=os.environ["R2_ENDPOINT_URL"],
        aws_access_key_id=os.environ["R2_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["R2_SECRET_ACCESS_KEY"],
        region_name="auto",
    )
    prefix = f"{MODEL_KEY}/{RUN_ID}"
    for local_path in RUN_DIR.rglob("*"):
        if local_path.is_file():
            key = f"{prefix}/{local_path.relative_to(RUN_DIR).as_posix()}"
            client.upload_file(str(local_path), os.environ["R2_BUCKET_NAME"], key)
    print(f"Uploaded to r2://{os.environ['R2_BUCKET_NAME']}/{prefix}/")
else:
    print("R2 upload skipped. Set UPLOAD_TO_R2=True after configuring Lightning secrets.")


## Interview checklist

Be ready to explain: why scaling is needed, why identifier codes are not quantities,
why high-cardinality fields are not blindly one-hot encoded, why accuracy is not the
main metric, and why the validation threshold is reused unchanged on the holdout.
